In [1]:
import pandas as pd
import numpy as np
import json
import cv2
from scipy.ndimage import gaussian_filter
import os

In [2]:
# 1. Paramètres
CSV_PATH    = "dataset_project_iapr2025/label_annotated.csv"   # votre export CSV de VIA
IMG_DIR     = "dataset_project_iapr2025/train"               # dossier de vos .JPG
OUT_DIR     = "density_maps/"              # dossier où sauver les .npy
SIGMA       = 100                     # écart-type de la gaussienne
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Crème_brûlée",
    "Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]

In [3]:
# 2. Préparez le mapping nom→indice
class2idx = {name:i for i,name in enumerate(CLASS_NAMES)}

In [4]:
# 3. Chargez tout le CSV en DataFrame
df = pd.read_csv(CSV_PATH)

In [11]:
import os
import json
import pandas as pd
import numpy as np
import cv2
from scipy.signal.windows import gaussian as scipy_gaussian

# 1. Paramètres
CSV_PATH    = "dataset_project_iapr2025/label_annotated.csv"   # votre export CSV de VIA
IMG_DIR     = "dataset_project_iapr2025/train"                # dossier de vos .JPG
OUT_DIR     = "density_maps/"                                 # dossier où sauver les .npz
SIGMA       = 150                                             # écart-type de la gaussienne
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Crème_brûlée",
    "Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]

# 2. Mapping nom → indice
class2idx = {name: i for i, name in enumerate(CLASS_NAMES)}

# 3. Précompute the Gaussian kernel
kern_size = int(6 * SIGMA) | 1   # kernel size, ensure odd
g1d = scipy_gaussian(kern_size, SIGMA)
kernel2d = np.outer(g1d, g1d)
kernel2d /= kernel2d.sum()       # normalize sum to 1
half_k = kern_size // 2

# 4. Charger le CSV
df = pd.read_csv(CSV_PATH)

# 5. Créer le dossier de sortie
os.makedirs(OUT_DIR, exist_ok=True)

# 6. Boucle sur chaque image
for fname, group in df.groupby("filename"):
    # 6.1 Lire l'image pour connaître ses dimensions
    img_path = os.path.join(IMG_DIR, fname)
    img = cv2.imread(img_path)
    if img is None:
        print(f"[ERROR] Impossible de lire l'image : {img_path}")
        continue
    H, W = img.shape[:2]
    
    # 6.2 Initialiser le tenseur de density maps
    density = np.zeros((H, W, len(CLASS_NAMES)), dtype=np.float32)
    
    # 6.3 Pour chaque point annoté, scatter-add le kernel
    for _, row in group.iterrows():
        shape_attr = json.loads(row["region_shape_attributes"])
        x = int(shape_attr["cx"])
        y = int(shape_attr["cy"])
        choco_type = json.loads(row["region_attributes"])["Choco_Type"]
        c = class2idx.get(choco_type, None)
        if c is None:
            continue
        
        # bounds for scatter-add
        y0 = max(0,   y - half_k)
        y1 = min(H,   y + half_k + 1)
        x0 = max(0,   x - half_k)
        x1 = min(W,   x + half_k + 1)
        ky0 = half_k - (y - y0)
        ky1 = half_k + (y1 - y)
        kx0 = half_k - (x - x0)
        kx1 = half_k + (x1 - x)
        
        density[y0:y1, x0:x1, c] += kernel2d[ky0:ky1, kx0:kx1]
    
    # 6.4 Conversion en float16 pour réduire la taille
    density = density.astype(np.float16)
    
    # 6.5 Sauvegarder au format compressé .npz
    out_path = os.path.join(OUT_DIR, fname.replace(".JPG", ".npz"))
    np.savez_compressed(out_path, density=density)
    
    print(f"Saved compressed density map for {fname} → {out_path}")


Saved compressed density map for L1000756.JPG → density_maps/L1000756.npz


KeyboardInterrupt: 

In [12]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ---- CONFIG ----
IMG_DIR   = "dataset_project_iapr2025/train"       # dossier de vos .JPG
DENS_DIR  = "density_maps"      # dossier de vos .npy
OUT_DIR   = "overlayed_maps"     # où sauver les PNG
FNAME     = "L1000756"     # nom de base (sans extension)
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black",
    "Amandina","Crème_brûlée","Triangolo","Tentation_noir",
    "Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]

os.makedirs(OUT_DIR, exist_ok=True)

# 1) Lire l'image
img_path = os.path.join(IMG_DIR, f"{FNAME}.JPG")
img_bgr  = cv2.imread(img_path)
if img_bgr is None:
    raise FileNotFoundError(f"Image not found: {img_path}")
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# 2) Charger la density-map
dens_path = os.path.join(DENS_DIR, f"{FNAME}.npz")
# Chargez le fichier compressé
data = np.load(dens_path)     # data est un NpzFile
# Récupérez votre array nommé 'density'
density = data['density']     # maintenant density est un ndarray (H, W, 13)

H,W,_ = density.shape

# 3) Créer un overlay par canal
for c, cls in enumerate(CLASS_NAMES):
    D = density[:,:,c]
    if D.max() == 0:
        print(f"[WARN] pas de points pour la classe {cls}, je skip.")
        continue

    # a) Normalisation (0→1)
    Dn = D / D.max()

    # b) Générer la figure
    plt.figure(figsize=(8,8))
    plt.imshow(img)
    # Afficher la heatmap avec colormap, alpha, et interpolation nearest
    plt.imshow(Dn, cmap="jet", alpha=0.5, interpolation="nearest")
    plt.title(f"{cls} ({D.sum():.2f} pts)")
    plt.axis("off")

    # c) Sauvegarde
    out_png = os.path.join(OUT_DIR, f"{FNAME}_{cls}.png")
    plt.savefig(out_png, bbox_inches="tight", pad_inches=0)
    plt.close()

print("Overlays générés dans", OUT_DIR)


[WARN] pas de points pour la classe Jelly_Milk, je skip.
[WARN] pas de points pour la classe Jelly_Black, je skip.
[WARN] pas de points pour la classe Amandina, je skip.
[WARN] pas de points pour la classe Crème_brûlée, je skip.
[WARN] pas de points pour la classe Tentation_noir, je skip.
[WARN] pas de points pour la classe Comtesse, je skip.
[WARN] pas de points pour la classe Noir_authentique, je skip.
[WARN] pas de points pour la classe Passion_au_lait, je skip.
[WARN] pas de points pour la classe Arabia, je skip.
Overlays générés dans overlayed_maps
